In [ ]:
workspace = "urban-analytics"
window_start = "2024-01-01"
window_end = "2024-06-30"


In [ ]:
from datetime import date

from urban_analytics.platform import Platform, Window

platform = Platform.fabric(workspace, Window(date.fromisoformat(window_start), date.fromisoformat(window_end)))


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

urban = spark.table(platform.gold.table("fact_urban_daily")).orderBy("date_key").toPandas()
urban["full_date"] = pd.to_datetime(urban["full_date"])


In [ ]:
figure, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
axes[0].plot(urban["full_date"], urban["trips"], color="#0d6a6a")
axes[0].set_ylabel("trips per day")
axes[1].plot(urban["full_date"], urban["avg_pm25"], color="#8f4f00", label="PM2.5")
axes[1].plot(urban["full_date"], urban["avg_no2"] * 1000, color="#1c6a40", label="NO2 (ppb)")
axes[1].legend(loc="upper right")
axes[1].set_ylabel("air quality")
axes[2].plot(urban["full_date"], urban["revenue_usd"], color="#0d6a6a", label="USD")
axes[2].plot(urban["full_date"], urban["revenue_eur"], color="#4bb6b0", label="EUR")
axes[2].legend(loc="upper right")
axes[2].set_ylabel("daily revenue")
figure.suptitle("Mobility, air quality and revenue on one calendar")
figure.tight_layout()


In [ ]:
correlation = urban[["trips", "avg_pm25", "avg_no2", "avg_o3", "temperature_mean_c", "precipitation_mm", "usd_per_eur"]].corr()
display(spark.createDataFrame(correlation.reset_index().rename(columns={"index": "metric"})))


In [ ]:
hourly = spark.table(platform.gold.table("fact_taxi_hourly")).groupBy("hour").avg("trips").orderBy("hour").toPandas()
pollution_hourly = (
    spark.table(platform.gold.table("fact_air_quality_hourly"))
    .filter("parameter = 'no2'")
    .groupBy("hour")
    .avg("avg_value")
    .orderBy("hour")
    .toPandas()
)
figure, axis = plt.subplots(figsize=(12, 4))
axis.bar(hourly["hour"], hourly["avg(trips)"], color="#4bb6b0", label="avg trips")
axis.set_ylabel("trips")
twin = axis.twinx()
twin.plot(pollution_hourly["hour"], pollution_hourly["avg(avg_value)"], color="#8f4f00", marker="o", label="NO2")
twin.set_ylabel("NO2")
axis.set_xticks(range(24))
axis.set_title("Time of day: taxi demand against NO2")
figure.tight_layout()


In [ ]:
strongest = (
    spark.table(platform.gold.table("fact_zone_station_correlation"))
    .join(spark.table(platform.gold.table("dim_zone")), "zone_key")
    .join(spark.table(platform.gold.table("dim_station")), "station_key")
    .select("zone_name", "borough", "station_name", "parameter", "pearson_r", "days", "avg_daily_trips")
    .orderBy("pearson_r", ascending=False)
)
display(strongest.limit(20))
